# Memproses Sekuens dengan Konvolusi 1D

**Buku:** *Deep Learning dengan Python* — Bab 6, Deep Learning untuk Teks dan Sekuens

Conv1D sebagai alternatif cepat RNN untuk teks, keterbatasannya pada deret waktu, dan kombinasi CNN→RNN untuk sekuens panjang.

> Diadaptasi dari notebook pendamping *Deep Learning with Python* (F. Chollet, Manning; kode berlisensi MIT) dan diperbarui ke **Keras 3** untuk buku *Deep Learning dengan Python: Konsep, Kode, dan Penerapan Modern* (B. S. Negara).

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+, Keras 3. Kode bersifat *backend-agnostik*: dapat dijalankan dengan backend TensorFlow, JAX, atau PyTorch.

**Perubahan utama dari versi asli (Keras 2.0.8):**
- **Bug asli diperbaiki:** model IMDB berakhir dengan `Dense(1)` tanpa sigmoid padahal memakai `binary_crossentropy` (keluaran harus berupa peluang).
- `RMSprop(lr=...)` → `RMSprop(learning_rate=...)`; `input_length` pada `Embedding` dihapus.
- Generator Jena → `timeseries_dataset_from_array`.

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

## 1. Conv1D untuk sentimen IMDB

In [ ]:
from keras.datasets import imdb

MAKS_KATA = 10000     # kosakata: 10.000 kata paling sering
PANJANG = 500         # potong ulasan setelah 500 kata
(x_latih, y_latih), (x_uji, y_uji) = imdb.load_data(num_words=MAKS_KATA)
x_latih = keras.utils.pad_sequences(x_latih, maxlen=PANJANG)
x_uji = keras.utils.pad_sequences(x_uji, maxlen=PANJANG)
print(x_latih.shape, x_uji.shape)

In [ ]:
import matplotlib.pyplot as plt

def plot_riwayat(riwayat, metrik="accuracy", judul=""):
    h = riwayat.history
    epoch = range(1, len(h["loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(epoch, h[metrik], "o", ms=3, label="Latih")
    ax[0].plot(epoch, h["val_" + metrik], "-", label="Validasi")
    ax[0].set(xlabel="Epoch", ylabel=metrik, title=judul)
    ax[1].plot(epoch, h["loss"], "o", ms=3, label="Latih")
    ax[1].plot(epoch, h["val_loss"], "-", label="Validasi")
    ax[1].set(xlabel="Epoch", ylabel="Loss", title=judul)
    for a in ax: a.legend()
    plt.tight_layout(); plt.show()

from keras import layers

model = keras.Sequential([
    keras.Input(shape=(PANJANG,), dtype="int64"),
    layers.Embedding(MAKS_KATA, 128),
    layers.Conv1D(32, 7, activation="relu"),
    layers.MaxPooling1D(5),
    layers.Conv1D(32, 7, activation="relu"),
    layers.GlobalMaxPooling1D(),
    layers.Dense(1, activation="sigmoid"),       # asli: tanpa sigmoid (bug)
])
model.summary()
model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=1e-4),
              loss="binary_crossentropy", metrics=["accuracy"])
riwayat = model.fit(x_latih, y_latih, epochs=10, batch_size=128, validation_split=0.2)
plot_riwayat(riwayat, judul="Conv1D pada IMDB")

## 2. Conv1D pada deret waktu Jena
Convnet tidak peka posisi waktu, sehingga sulit mengalahkan baseline akal sehat.

In [ ]:
import zipfile, numpy as np

URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"
jalur_zip = keras.utils.get_file("jena_climate_2009_2016.csv.zip", origin=URL)
with zipfile.ZipFile(jalur_zip) as zf:
    teks = zf.read("jena_climate_2009_2016.csv").decode("utf-8")

baris = teks.strip().split("\n")
header = [h.strip('"') for h in baris[0].split(",")]
baris = baris[1:]
data_mentah = np.array([[float(v) for v in b.split(",")[1:]] for b in baris], dtype="float32")
suhu = data_mentah[:, 1].copy()                     # kolom "T (degC)"
print(len(baris), "baris |", header[1:4], "...")

N_LATIH, N_VAL = 200_000, 100_000                    # sama dengan pembagian asli
rerata = data_mentah[:N_LATIH].mean(axis=0)
simpangan = data_mentah[:N_LATIH].std(axis=0)
data = (data_mentah - rerata) / simpangan            # masukan dinormalisasi; target tetap °C

In [ ]:
def buat_dataset(laju_sampel=6, panjang=120, batch=256):
    """Masukan: `panjang` titik dengan jarak `laju_sampel` langkah (10 menit/langkah).
    Target: suhu 24 jam setelah titik terakhir."""
    tunda = laju_sampel * (panjang - 1) + 144        # 144 langkah = 24 jam
    umum = dict(sampling_rate=laju_sampel, sequence_length=panjang, batch_size=batch)
    latih = keras.utils.timeseries_dataset_from_array(
        data[:-tunda], targets=suhu[tunda:], shuffle=True,
        start_index=0, end_index=N_LATIH, **umum)
    val = keras.utils.timeseries_dataset_from_array(
        data[:-tunda], targets=suhu[tunda:],
        start_index=N_LATIH, end_index=N_LATIH + N_VAL, **umum)
    uji = keras.utils.timeseries_dataset_from_array(
        data[:-tunda], targets=suhu[tunda:], start_index=N_LATIH + N_VAL, **umum)
    return latih, val, uji

ds_latih, ds_val, ds_uji = buat_dataset()
for sampel, target in ds_latih.take(1):
    print("Sampel:", sampel.shape, "| target:", target.shape)

In [ ]:
def mae_baseline(dataset):
    total, n = 0.0, 0
    for s, t in dataset:
        s = keras.ops.convert_to_numpy(s)
        total += np.abs(s[:, -1, 1] * simpangan[1] + rerata[1] - keras.ops.convert_to_numpy(t)).sum()
        n += len(t)
    return total / n

def latih_jena(model, ds_l, ds_v, epochs=20, judul=""):
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    h = model.fit(ds_l, epochs=epochs, validation_data=ds_v).history
    plt.plot(h["mae"], "o", ms=3, label="Latih"); plt.plot(h["val_mae"], "-", label="Validasi")
    plt.axhline(mae_baseline(ds_v), color="gray", ls="--", label="baseline")
    plt.xlabel("Epoch"); plt.ylabel("MAE (°C)"); plt.title(judul); plt.legend(); plt.show()
    print(f"MAE validasi terbaik: {min(h['val_mae']):.2f} °C")

latih_jena(keras.Sequential([keras.Input(shape=(None, data.shape[-1])),
                             layers.Conv1D(32, 5, activation="relu"), layers.MaxPooling1D(3),
                             layers.Conv1D(32, 5, activation="relu"), layers.MaxPooling1D(3),
                             layers.Conv1D(32, 5, activation="relu"),
                             layers.GlobalMaxPooling1D(), layers.Dense(1)]),
           ds_latih, ds_val, judul="Conv1D saja")

## 3. CNN → GRU untuk sekuens yang lebih rapat
Resolusi digandakan (satu titik per 30 menit); Conv1D memendekkan sekuens sebelum GRU.

In [ ]:
ds_latih3, ds_val3, ds_uji3 = buat_dataset(laju_sampel=3, panjang=240)
latih_jena(keras.Sequential([keras.Input(shape=(None, data.shape[-1])),
                             layers.Conv1D(32, 5, activation="relu"), layers.MaxPooling1D(3),
                             layers.Conv1D(32, 5, activation="relu"),
                             layers.GRU(32, dropout=0.1, recurrent_dropout=0.5),
                             layers.Dense(1)]),
           ds_latih3, ds_val3, judul="Conv1D → GRU")